# Remblais

**Do not cross-fade the pixels. Move the mass.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Unsuspicious-Industries/remblais/blob/main/demo.ipynb)

Remblais computes an unbalanced optimal transport coupling, decomposes it onto local grid paths, and renders those conservative moves as an image morph.

In [ ]:
# @title Install Remblais
import os
import subprocess
import sys

os.environ.setdefault("CC", "clang")
if "google.colab" in sys.modules:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/Unsuspicious-Industries/remblais",
        ],
        check=True,
    )

In [ ]:
# @title Transport controls
resolution = 24  # @param {type:"slider", min:16, max:40, step:8}
X = 1.0  # @param {type:"number"}
Y = 10000000000.0  # @param {type:"number"}
epsilon = 1.0  # @param {type:"number"}
sinkhorn_steps = 150  # @param {type:"slider", min:50, max:300, step:50}
frame_count = 24  # @param {type:"slider", min:8, max:48, step:4}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as DisplayImage
from IPython.display import display

from remblais import morph, save_video

n = resolution
source = np.zeros((n, n, 3), dtype=np.uint8)
source[: n // 4, : n // 4] = 255

target = np.zeros_like(source)
target[3 * n // 4 :, 3 * n // 4 :, 0] = 255
target[n // 2 : 3 * n // 4, 3 * n // 4 :, 1] = 255
target[3 * n // 4 :, n // 2 : 3 * n // 4, 2] = 255

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, image, title in zip(axes, (source, target), ("source", "target")):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Transport, route, render

The dense tinygrad step solves one entropic unbalanced OT problem per RGB channel. The CPU routing step rounds that coupling, factors each transport edge through a shortest Manhattan path, and records local mass transfers.

In [ ]:
result = morph(
    source,
    target,
    X=X,
    Y=Y,
    epsilon=epsilon,
    num_iters=sinkhorn_steps,
    n_frames=frame_count,
)
frames = result.frames_np()
save_video(frames, "remblais.gif", fps=12)
display(DisplayImage(filename="remblais.gif"))

In [ ]:
indices = np.linspace(0, len(frames) - 1, 8).round().astype(int)
fig, axes = plt.subplots(1, len(indices), figsize=(14, 2))
for ax, i in zip(axes, indices):
    ax.imshow(frames[i])
    ax.axis("off")
plt.tight_layout()
plt.show()

## What is contributed here?

A coupling is static. Remblais lifts it into dynamics. Integer source mass is conserved while it moves through local grid edges: $x \leftarrow x + m(e_j-e_i)$. Any unbalanced marginal error remains explicit as a final residual $r=b-\tilde b$, which makes the endpoint exact.

Try increasing `X`. Long transport becomes expensive, so more of the change moves into the final cut-and-fill residual. Increase `epsilon` to spread the coupling. Routing samples new shortest paths on each run.

The plan is dense and costs $O((HW)^2)$ memory. Small images are the point.